# 01 — Data processing

Gabriela's scope: audit the Roboflow YOLO export `garbage-classification-3`, clean annotations, and build our own train/val/test split (test stays held-out).

Raw data is **not modified**. Audit manifests go to `data/interim/`; the training-ready dataset goes to `data/processed/`.

CLI equivalent:

```bash
uv run python scripts/process_data.py
```


In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import logging
from pathlib import Path

from vpc2.data import io
from vpc2.data.processing import collect_yolo_files, run_pipeline

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
INTERIM_DIR = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed"

dataset_root = io.discover_raw_dataset(RAW_DIR)
yaml_payload = io.load_yolo_yaml(dataset_root / "data.yaml")
class_names = io.class_names_from_yaml(yaml_payload)
images, labels = collect_yolo_files(dataset_root)

print(f"dataset_root: {dataset_root}")
print(f"classes ({len(class_names)}): {class_names}")
print(f"images: {len(images)}  labels: {len(labels)}")

dataset_root: /Users/alejandrovalle/Desktop/Posgrado/CEIA/5to_bimestre/VPC2/VPC2_TP/data/raw
classes (6): ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']
images: 10464  labels: 10464


## Pipeline: audit → clean → split → write

If the unzip is incomplete (for example missing `train/labels` or `valid/`), the report flags it and only uses valid image+label pairs.


In [6]:
report = run_pipeline(
    raw_dir=RAW_DIR,
    interim_dir=INTERIM_DIR,
    processed_dir=PROCESSED_DIR,
    seed=42,
    ratios=(0.70, 0.20, 0.10),
)

print(
    {
        "paired_kept": report.paired_kept,
        "corrupt": report.images_corrupt,
        "orphan_images": report.orphan_images,
        "orphan_labels": report.orphan_labels,
        "empty_labels": report.empty_labels,
        "boxes_clipped": report.boxes_clipped,
        "duplicates_removed": report.duplicates_removed,
        "split": {
            "train": report.split_train,
            "val": report.split_val,
            "test": report.split_test,
        },
        "class_histogram": report.class_histogram,
    }
)

INFO vpc2.data.processing: Using raw dataset at /Users/alejandrovalle/Desktop/Posgrado/CEIA/5to_bimestre/VPC2/VPC2_TP/data/raw
Write test: 100%|██████████| 1045/1045 [00:00<00:00, 3370.15img/s]
INFO vpc2.data.processing: Dataset root: /Users/alejandrovalle/Desktop/Posgrado/CEIA/5to_bimestre/VPC2/VPC2_TP/data/raw
INFO vpc2.data.processing: Classes (6): ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']
INFO vpc2.data.processing: Images scanned=10464 valid=10464 corrupt=0 converted=0
INFO vpc2.data.processing: Labels scanned=10464 orphan_images=0 orphan_labels=0 empty=0
INFO vpc2.data.processing: Boxes raw=74090 kept=74073 clipped=0 dropped=17 dups_removed=0 malformed=0 invalid_class=0
INFO vpc2.data.processing: Paired samples kept=10464 | split train/val/test = 7326/2093/1045
INFO vpc2.data.processing: Class histogram (instances): {'BIODEGRADABLE': 45395, 'CARDBOARD': 4696, 'GLASS': 7809, 'METAL': 5841, 'PAPER': 4387, 'PLASTIC': 5945}
INFO vpc2.data.processing: Process

{'paired_kept': 10464, 'corrupt': 0, 'orphan_images': 0, 'orphan_labels': 0, 'empty_labels': 0, 'boxes_clipped': 0, 'duplicates_removed': 0, 'split': {'train': 7326, 'val': 2093, 'test': 1045}, 'class_histogram': {'BIODEGRADABLE': 45395, 'CARDBOARD': 4696, 'GLASS': 7809, 'METAL': 5841, 'PAPER': 4387, 'PLASTIC': 5945}}


## Handoff for the rest of the team

- Alejandro (augmentation): `data/processed/` — do **not** augment the test split.
- Julia (train): `data/processed/data.yaml`


In [8]:
report_path = INTERIM_DIR / "audit" / "report.json"
yaml_path = PROCESSED_DIR / "data.yaml"

print("audit report:", report_path if report_path.exists() else "missing")
print("data.yaml:")
print(yaml_path.read_text(encoding="utf-8") if yaml_path.exists() else "missing")

if report.warnings:
    print("\nWarnings:")
    for warning in report.warnings:
        print("-", warning)

audit report: /Users/alejandrovalle/Desktop/Posgrado/CEIA/5to_bimestre/VPC2/VPC2_TP/data/interim/audit/report.json
data.yaml:
path: /Users/alejandrovalle/Desktop/Posgrado/CEIA/5to_bimestre/VPC2/VPC2_TP/data/processed
train: images/train
val: images/val
test: images/test
nc: 6
names:
  0: BIODEGRADABLE
  1: CARDBOARD
  2: GLASS
  3: METAL
  4: PAPER
  5: PLASTIC

